# Montage Maker

**Tap the play button below.** A new montage shows up in **My Drive / Montage Maker / Montages**.
Tap it again for another one.

What happens when you tap it:
1. Google asks to connect your Drive. Allow it.
2. It asks for your Apple ID to pull videos from iCloud too. Press Enter to skip and use only your Drive.
3. It looks through your videos on Google's free computer (not your phone) and skips screenshots, screen recordings, and anything with text in the top half, where your caption goes.
4. It finds the best moment in each, cuts clips of different lengths (short opener, then a mix), and joins them into a vertical video of about 20 seconds.
5. Only the finished montage is saved to your Drive.

Free.


In [ ]:
# Montage Maker: press the play button. A new montage lands in My Drive/Montage Maker/Montages.
# Run it again for another one.

import json, math, random, shutil, subprocess, sys, datetime, getpass, urllib.request
from pathlib import Path

MY_DRIVE = Path('/content/drive/MyDrive')
FOLDER = MY_DRIVE / 'Montage Maker'
OUT_DIR = FOLDER / 'Montages'
WORK = Path('/content/work')
ICLOUD = Path('/content/icloud')
MODELS = Path('/content/models')

WIDTH, HEIGHT, FPS = 1080, 1920, 30
VIDEO_TYPES = {'.mp4', '.mov', '.m4v'}
LIBRARY_START = datetime.date(2012, 1, 1)   # earliest date to look for videos in iCloud
WINDOW_DAYS = 14                            # each run grabs videos from a random 2-week stretch

# Clip lengths (see IDEAS.md in the repo for the sources behind these)
TARGET_SECONDS = 20    # total length; 15 to 30 s Reels showed the best engagement
MAX_CLIPS = 12
MEDIAN_CLIP = 2.0      # typical clip length; lengths vary around it, never all the same
SPREAD = 0.45          # how much they vary (log-normal, like shot lengths in real films)
SHORTEST, LONGEST = 1.0, 4.5
HOOK_LONGEST = 1.5     # the first clip is short and busy, to hook in the first seconds
MIN_SECONDS = 1.0      # videos shorter than this are skipped

TEXT_MODEL_URL = ('https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/'
                  'text_detection_ppocr/text_detection_en_ppocrv3_2023may.onnx')


def run(cmd):
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{cmd[0]} failed:\n{r.stderr[-1500:]}")
    return r.stdout


def probe(path):
    out = run(['ffprobe', '-v', 'error', '-select_streams', 'v:0',
               '-show_entries', 'stream=width,height,color_transfer:format=duration',
               '-of', 'json', path])
    j = json.loads(out)
    s = (j.get('streams') or [{}])[0]
    return {'duration': float(j.get('format', {}).get('duration') or 0),
            'has_video': bool(s.get('width')),   # some .mp4 files are sound only
            'hdr': s.get('color_transfer') in ('smpte2084', 'arib-std-b67')}


def has_filter(name):
    return f' {name} ' in run(['ffmpeg', '-hide_banner', '-filters'])


# ---------- getting the videos ----------

def pull_from_icloud():
    """Copy videos from a random 2-week stretch of your iCloud library onto this free
    Google computer. Not your phone, not your Drive. Wiped when the session ends."""
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'icloudpd'], check=True)
    from icloudpd.cli import cli

    id_file = FOLDER / '.apple-id.txt'
    apple_id = id_file.read_text().strip() if id_file.exists() else input(
        'Apple ID email (press Enter to skip iCloud): ').strip()
    if not apple_id:
        return []
    id_file.write_text(apple_id)
    password = getpass.getpass('Apple ID password (not saved). Press Enter to skip iCloud: ')
    if not password:
        return []

    for attempt in range(6):
        span = (datetime.date.today() - LIBRARY_START).days - WINDOW_DAYS
        start = LIBRARY_START + datetime.timedelta(days=random.randint(0, max(span, 0)))
        end = start + datetime.timedelta(days=WINDOW_DAYS)
        dest = ICLOUD / start.isoformat()
        dest.mkdir(parents=True, exist_ok=True)
        print(f'Looking for videos from {start:%b %d, %Y} to {end:%b %d, %Y}...')
        sys.argv = ['icloudpd', '--username', apple_id,
                    '--password', password, '--password-provider', 'parameter',
                    '--mfa-provider', 'console',
                    '--cookie-directory', str(FOLDER / '.icloud-login'),
                    '--directory', str(dest), '--folder-structure', 'none',
                    '--skip-photos', '--skip-live-photos',
                    '--skip-created-before', start.isoformat(),
                    '--skip-created-after', end.isoformat(),
                    '--no-progress-bar', '--log-level', 'error']
        try:
            cli()
        except SystemExit:
            pass
        found = [p for p in dest.iterdir() if p.suffix.lower() in VIDEO_TYPES]
        if len(found) >= 2:
            print(f'Got {len(found)} videos.')
            return found
        print('Not enough videos there, trying another stretch.')
    return []


def videos_in_drive():
    """Every video anywhere in your Google Drive, except finished montages."""
    if not MY_DRIVE.exists():
        return []
    return [p for p in MY_DRIVE.rglob('*')
            if p.suffix.lower() in VIDEO_TYPES and OUT_DIR not in p.parents]


# ---------- looking at each video ----------

LOOK_SECONDS = 60      # only the first minute of a long video is looked at, to keep it quick
_detector = None


def text_detector():
    """A small free text-finding model (OpenCV model zoo, PP-OCRv3, Apache 2.0, 2.4 MB).
    Downloaded once per session onto Google's computer."""
    global _detector
    if _detector is None:
        import cv2
        MODELS.mkdir(parents=True, exist_ok=True)
        path = MODELS / 'text_detection.onnx'
        if not path.exists():
            urllib.request.urlretrieve(TEXT_MODEL_URL, path)
        m = cv2.dnn.TextDetectionModel_DB(cv2.dnn.readNet(str(path)))
        m.setBinaryThreshold(0.3)
        m.setPolygonThreshold(0.5)
        m.setUnclipRatio(2.0)
        m.setMaxCandidates(200)
        m.setInputParams(1.0 / 255.0, (736, 736), (122.68, 116.67, 104.0))
        _detector = m
    return _detector


def on_screen_text(video, duration):
    """Which parts of the video to avoid because of text. Looks at the video already
    cropped to vertical, the way it will appear in the montage.

    Avoided: screenshots and screen recordings (lots of flat text), and text that sits
    still in the top half, where the caption goes. Allowed: text on real things, like a
    label on a mug, that moves or tilts with the camera.

    Returns (spots to avoid, True if the whole video looks like a screen)."""
    import cv2, numpy as np
    rate = 1 if duration >= 3 else 3          # frames per second to check
    folder = WORK / (video.stem + '_frames')
    folder.mkdir(parents=True, exist_ok=True)
    run(['ffmpeg', '-hide_banner', '-v', 'error', '-t', LOOK_SECONDS, '-i', video,
         '-vf', f'fps={rate},scale={WIDTH}:{HEIGHT}:force_original_aspect_ratio=increase,'
                f'crop={WIDTH}:{HEIGHT},scale=540:960', folder / '%03d.jpg'])
    det = text_detector()
    frames = []                                # per frame: (level boxes, screen-like)
    for f in sorted(folder.glob('*.jpg')):
        img = cv2.imread(str(f))
        h, w = img.shape[:2]
        found, _ = det.detect(img)
        level, area = [], 0.0
        for poly in found:
            pts = np.array(poly, dtype=np.float32)
            angle = abs(cv2.minAreaRect(pts)[2]) % 90
            if min(angle, 90 - angle) < 5:     # flat, the way captions and screens are
                x, y, bw, bh = cv2.boundingRect(pts.astype(np.int32))
                level.append((x, y, x + bw, y + bh))
                area += bw * bh / (w * h)
        frames.append((level, len(level) >= 4 or area > 0.03, h))
    shutil.rmtree(folder, ignore_errors=True)

    def overlap(a, b):
        ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
        iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
        inter = ix * iy
        union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
        return inter / union if union else 0

    bad = {i for i, (_, screen, _) in enumerate(frames) if screen}
    for i in range(len(frames) - 1):           # still text in the top half
        top_now = [r for r in frames[i][0] if (r[1] + r[3]) / 2 < frames[i][2] / 2]
        if any(overlap(a, b) > 0.6 for a in top_now for b in frames[i + 1][0]):
            bad.update({i, i + 1})
    whole = bool(frames) and len(bad) > len(frames) / 2
    spots = [(i / rate - 0.5 / rate, i / rate + 0.5 / rate) for i in sorted(bad)]
    return spots, whole


def movement(video):
    """Movement and brightness 4 times a second: [(time, movement, brightness), ...]."""
    stats = WORK / (video.stem + '.stats.txt')
    run(['ffmpeg', '-hide_banner', '-v', 'error', '-t', LOOK_SECONDS, '-i', video,
         '-vf', f"fps=4,scale=160:-2,signalstats,metadata=print:file='{stats}'",
         '-an', '-f', 'null', '-'])
    frames = []
    for line in stats.read_text().splitlines():
        if line.startswith('frame:'):
            frames.append([float(line.split('pts_time:')[1]), 0.0, None])
        elif frames and 'signalstats.YDIF=' in line:
            frames[-1][1] = float(line.split('=')[1])
        elif frames and 'signalstats.YAVG=' in line:
            frames[-1][2] = float(line.split('=')[1])
    return [f for f in frames if f[2] is not None]


def best_start(frames, duration, length, blocked):
    """Best place to cut a clip of this length: lots of movement, not too dark or bright,
    no on-screen text. One of the top 3 spots at random, so each run can differ.
    Returns (start, score), or None if every spot has text."""
    latest = min(duration, LOOK_SECONDS) - length - 0.05
    if latest < 0:
        return None
    candidates = []
    for k in range(int(latest / 0.25) + 1):
        s = k * 0.25
        if any(s < b_end and s + length > b_start for b_start, b_end in blocked):
            continue
        window = [f for f in frames if s <= f[0] < s + length]
        if not window:
            candidates.append((0.0, s))
            continue
        motion = sum(f[1] for f in window) / len(window)
        light = sum(f[2] for f in window) / len(window)
        candidates.append((motion * (0.3 if light < 40 or light > 220 else 1.0), s))
    if not candidates:
        return None
    candidates.sort(reverse=True)
    top = []
    for score, s in candidates:
        if score < candidates[0][0] * 0.5:
            break
        if all(abs(s - o) >= length for _, o in top):
            top.append((score, s))
        if len(top) == 3:
            break
    score, s = random.choice(top)
    return s, score


# ---------- clip lengths ----------

def clip_lengths(clips):
    """Different lengths, never all the same. Drawn from a log-normal spread around
    MEDIAN_CLIP (how shot lengths fall in real films), then busier clips get shorter and
    calmer ones longer (in modern films shorter shots carry more motion), then scaled
    toward TARGET_SECONDS. The busiest clip opens the video, kept short, as the hook."""
    order = sorted(range(len(clips)), key=lambda i: clips[i]['score'])   # calm to busy
    rank = {i: (r / (len(clips) - 1) if len(clips) > 1 else 0.5) for r, i in enumerate(order)}
    raw = [random.lognormvariate(math.log(MEDIAN_CLIP), SPREAD) * (1.25 - 0.5 * rank[i])
           for i in range(len(clips))]
    scale = TARGET_SECONDS / sum(raw) if sum(raw) > TARGET_SECONDS else 1.0
    for i, c in enumerate(clips):
        c['length'] = max(SHORTEST, min(LONGEST, raw[i] * scale))
    hook = order[-1]
    clips[hook]['length'] = min(clips[hook]['length'], HOOK_LONGEST)
    clips.insert(0, clips.pop(hook))
    return clips


# ---------- clip and combine ----------

def make_clip(video, start, length, hdr, out):
    color = ('zscale=t=linear:npl=100,format=gbrpf32le,zscale=p=bt709,'
             'tonemap=tonemap=hable:desat=0,zscale=t=bt709:m=bt709:r=tv,format=yuv420p,'
             if hdr and has_filter('zscale') else '')
    vf = (f'{color}scale={WIDTH}:{HEIGHT}:force_original_aspect_ratio=increase,'
          f'crop={WIDTH}:{HEIGHT},fps={FPS},setsar=1,format=yuv420p')
    run(['ffmpeg', '-y', '-hide_banner', '-v', 'error', '-ss', f'{start:.2f}', '-i', video,
         '-t', f'{length:.2f}', '-vf', vf, '-an',
         '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '18', out])
    return out


def combine(clips, out):
    inputs = []
    for c in clips:
        inputs += ['-i', c]
    joins = ''.join(f'[{i}:v]' for i in range(len(clips)))
    run(['ffmpeg', '-y', '-hide_banner', '-v', 'error', *inputs,
         '-filter_complex', f'{joins}concat=n={len(clips)}:v=1:a=0[v]', '-map', '[v]',
         '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '20', '-pix_fmt', 'yuv420p',
         '-movflags', '+faststart', out])
    want = sum(probe(c)['duration'] for c in clips)
    got = probe(out)['duration'] if out.exists() else 0
    if got < want - 0.1 * len(clips):
        raise RuntimeError(f'Montage came out short: {got:.1f}s instead of {want:.1f}s')
    return out


def make_montage(videos):
    WORK.mkdir(parents=True, exist_ok=True)
    random.shuffle(videos)
    wanted = min(MAX_CLIPS, math.ceil(TARGET_SECONDS / MEDIAN_CLIP))
    clips = []
    for v in videos:                          # look at videos in random order
        if len(clips) == wanted:
            break
        try:
            info = probe(v)
        except Exception:
            continue
        if not info['has_video'] or info['duration'] < MIN_SECONDS:
            continue
        print(f'Looking at {v.name}...')
        frames = movement(v)
        blocked, screen = on_screen_text(v, info['duration'])
        if screen:
            print('   skipped: screenshot or screen recording')
            continue
        trial = min(MEDIAN_CLIP, info['duration'] - 0.1)
        pick = best_start(frames, info['duration'], trial, blocked)
        if pick is None:
            print('   skipped: on-screen text')
            continue
        clips.append({'video': v, 'info': info, 'frames': frames, 'blocked': blocked,
                      'start': pick[0], 'score': pick[1], 'length': trial})
    if not clips:
        raise SystemExit('No usable videos (too short, sound only, or covered in text).')

    clips = clip_lengths(clips)
    files = []
    for i, c in enumerate(clips):
        length = min(c['length'], c['info']['duration'] - 0.1)
        pick = best_start(c['frames'], c['info']['duration'], length, c['blocked'])
        start = pick[0] if pick else c['start']
        if not pick:
            length = min(length, c['info']['duration'] - c['start'] - 0.05)
        print(f'Clip {i + 1}: {length:.1f}s from {c["video"].name}')
        files.append(make_clip(c['video'], start, length, c['info']['hdr'], WORK / f'clip{i}.mp4'))
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    out = OUT_DIR / f"montage-{datetime.datetime.now():%Y-%m-%d-%H%M%S}.mp4"
    combine(files, out)
    shutil.rmtree(WORK, ignore_errors=True)
    return out


# ---------- go ----------

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount('/content/drive')
    FOLDER.mkdir(parents=True, exist_ok=True)
    videos = pull_from_icloud() + videos_in_drive()
    if not videos:
        raise SystemExit('No videos found in iCloud or in your Google Drive.')
    result = make_montage(videos)
    print(f'\nDone! Your montage is in My Drive/Montage Maker/Montages: {result.name}')
